### This notebook format data for training deepSCENIC

In [1]:
import pickle
import scanpy as sc
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import mudata as mu

Load scRNA-seq and scATAC-seq data

In [2]:
mdata = mu.read('/data/groups/vib.ai/stein.aerts/gpartel/data/test/ACC_GEX.h5mu')
mdata

/data/groups/vib.ai/stein.aerts/gpartel/software/miniconda3/envs/deepSCENIC/lib/python3.11/site-packages/mudata/_core/mudata.py:1598: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/data/groups/vib.ai/stein.aerts/gpartel/software/miniconda3/envs/deepSCENIC/lib/python3.11/site-packages/mudata/_core/mudata.py:1461: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


MuData object with n_obs × n_vars = 39470 × 569398
  2 modalities
    scRNA:	39470 x 24275
      obs:	'nCount_RNA', 'nFeature_RNA', 'nCount_ATAC', 'nFeature_ATAC', 'nCount_SCT', 'nFeature_SCT', 'class_Bakken_2022', 'subclass_Bakken_2022', 'cluster_Bakken_2022', 'species', 'sample'
      var:	'gene'
      obsm:	'pca', 'umap'
    scATAC:	39470 x 545123
      obs:	'cisTopic_nr_frag', 'cisTopic_log_nr_frag', 'cisTopic_nr_acc', 'cisTopic_log_nr_acc', 'sample_id', 'Unique_nr_frag', 'barcode', 'pycisTopic_leiden_10_0.6', 'pycisTopic_leiden_10_1.2', 'cluster_Bakken_2022', 'class_Bakken_2022', 'subclass_Bakken_2022'
      var:	'Chromosome', 'Start', 'End', 'Width', 'cisTopic_nr_frag', 'cisTopic_log_nr_frag', 'cisTopic_nr_acc', 'cisTopic_log_nr_acc'
      obsm:	'UMAP', 'tSNE'

In [3]:
DATA_PATH = '/data/groups/vib.ai/stein.aerts/gpartel/data/test/'
ad_rna = mdata.mod['scRNA']
ad_atac = mdata.mod['scATAC'] # This must be imputed accessibility from pyCisTopic

Filter and normalize scRNA-seq

In [4]:
sc.pp.filter_genes(ad_rna, min_cells=10)

In [5]:
sc.pp.normalize_total(ad_rna)
sc.pp.log1p(ad_rna)

Remove genes that have with zero variance

In [8]:
ad_rna = ad_rna[:,ad_rna.X.toarray().std(0)!=0]

Filter and normalize scATAC-seq data

In [9]:
# Filter region that are present in less then 10% of the cell for each cell type

regs = []
for ct in tqdm(ad_atac.obs.subclass_Bakken_2022.unique()):
    ad_ct = ad_atac[ad_atac.obs.subclass_Bakken_2022==ct].copy()
    sc.pp.filter_genes(ad_ct, min_cells=ad_ct.shape[0]*0.1)
    regs.append(ad_ct.var_names)

regs = [x.to_list() for x in regs]    
regs = list(set(sum(regs,[])))
len(regs)

ad_atac = ad_atac[:, regs]

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 19/19 [02:14<00:00,  7.07s/it]


In [10]:
ad_atac.region_names = ad_atac.var_names

In [11]:
TFs = pd.read_csv("/data/groups/vib.ai/stein.aerts/gpartel/resources/allTFs_mm.txt", header=None)[0].to_list()

In [12]:
# Subset TFs present in RNA data
TFs = ad_rna.var_names[ad_rna.var_names.isin(TFs)]
len(TFs)

1390

Get regions per gene (up to 1 Mb up and down stream of the gene)

In [ ]:
# This code is taken and adapted from scenicplus because it was not working anymore. But Seppe might have fixed it

from typing import Set, Tuple, Union
import logging
import sys
from typing import Set, Tuple, Union
import pybiomart as pbm
import xml.etree.ElementTree as xml_tree
import re
import requests
import time
import sys

_NCBI_MAX_RETRIES = 3

class MaxNCBIRetriesReached(Exception):
    def __init__(self):
        super().__init__(f"Data not found after {_NCBI_MAX_RETRIES} retries.")

class NCBISearchNotFound(Exception):
    def __init__(self, search_term, url):
        super().__init__(f"Could not find {search_term} on {url}")

def download_gene_annotation_and_chromsizes(
        species: str,
        biomart_host: str,
        use_ucsc_chromosome_style: bool = True
        ) -> Union[pd.DataFrame, Tuple[pd.DataFrame, pd.DataFrame]]:
    """
    Download gene annotation for specific species.
    This function needs access to the internet.

    Parameters
    ----------
    species: str
        species for which to get the gene annotation, e.g. "hsapiens"
    biomart_host: str
        url for the biomart host to use. Please make sure that the host
        used matches with the annotation used for running SCENIC+.

    Returns
    -------
    A pandas DataFrame with following columns:
        Chromosome, Start, End, Strand, Gene, Transcription_Start_Site, Transcript_type
    The dataframe is subsetted for protein coding genes. 
    """
    # Create logger
    level = logging.INFO
    format = '%(asctime)s %(name)-12s %(levelname)-8s %(message)s'
    handlers = [logging.StreamHandler(stream=sys.stdout)]
    logging.basicConfig(level=level, format=format, handlers=handlers)
    log = logging.getLogger('Download gene annotation')

    dataset_name = f"{species}_gene_ensembl"
    server = pbm.Server(host=biomart_host, use_cache=False)
    mart = server["ENSEMBL_MART_ENSEMBL"]
    if dataset_name not in mart.list_datasets()['name'].to_numpy():
        raise ValueError(
            f"The dataset name: {dataset_name} could not be found in biomart. "
            + "Check species name or consider manually providing gene annotations!")
    dataset = mart[dataset_name]

    external_gene_name_query = "external_gene_name" \
        if "external_gene_name" in dataset.attributes.keys() else "hgnc_symbol"
    transcription_start_site_query = "transcription_start_site" \
        if "transcription_start_site" in dataset.attributes.keys() else "transcript_start"

    annot = dataset.query(attributes=[
        "chromosome_name", "start_position", "end_position",
        "strand", external_gene_name_query,
        transcription_start_site_query, "transcript_biotype"
    ])
    annot.columns = ["Chromosome", "Start", "End", "Strand",
                     "Gene", "Transcription_Start_Site", "Transcript_type"]
    annot = annot[annot.Transcript_type == "protein_coding"].copy()
    annot['Strand'] = ['+' if strand == 1 else '-' for strand in annot['Strand']]

    # --- get assembly information from NCBI ---
    try:
        _regex_display_name = re.search(r'\((.*?)\)', dataset.display_name)
        if _regex_display_name is None:
            raise ValueError("Could not find assembly from biomart query display name.")
        ncbi_search_term = _regex_display_name.group(1)  # e.g., "GRCh38.p14"
        log.info(f"Using genome: {ncbi_search_term}")

        # Helper: robust GET with retries
        def _get_with_retries(url: str, params: dict = None) -> requests.Response:
            tries = 0
            resp = requests.get(url, params=params)
            while not resp.ok and tries < _NCBI_MAX_RETRIES:
                time.sleep(0.5)  # be kind to NCBI
                resp = requests.get(url, params=params)
                tries += 1
            if (tries == _NCBI_MAX_RETRIES) and not resp.ok:
                raise MaxNCBIRetriesReached
            return resp

        # 1) Try genome DB first
        esearch_genome_resp = _get_with_retries(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
            params={"db": "genome", "term": ncbi_search_term}
        )
        _IdList_element = xml_tree.fromstring(esearch_genome_resp.content).find('IdList')
        _Id_element = _IdList_element.find('Id') if _IdList_element is not None else None

        if _Id_element is not None:
            # genome → esummary(genome) → AssemblyID
            ncbi_genome_id = _Id_element.text
            log.info(f"Found corresponding genome Id {ncbi_genome_id} on NCBI")
            esummary_genome_resp = _get_with_retries(
                "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi",
                params={"db": "genome", "id": ncbi_genome_id}
            )
            _DocSum_element = xml_tree.fromstring(esummary_genome_resp.content).find('DocSum')
            if _DocSum_element is None:
                raise NCBISearchNotFound(
                    search_term='DocSum',
                    url=str(esummary_genome_resp.url)
                )
            _DocSumItem_elements = _DocSum_element.findall('Item')
            if not _DocSumItem_elements or not all(["Name" in x.attrib for x in _DocSumItem_elements]):
                raise NCBISearchNotFound(
                    search_term="Item/Name",
                    url=str(esummary_genome_resp.url)
                )
            _DocSumItem_dict = {x.attrib["Name"]: x.text for x in _DocSumItem_elements}
            if "AssemblyID" not in _DocSumItem_dict:
                raise NCBISearchNotFound(
                    search_term="AssemblyID",
                    url=str(esummary_genome_resp.url)
                )
            ncbi_assembly_id = _DocSumItem_dict["AssemblyID"]
        else:
            # 2) Fallback: search assembly DB directly with the assembly name
            log.info(f"No genome Id found for term '{ncbi_search_term}'. Trying assembly DB…")
            esearch_assembly_resp = _get_with_retries(
                "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
                # The fielded term improves precision but is optional
                params={"db": "assembly", "term": f"{ncbi_search_term}[Assembly Name]"}
            )
            _IdList_element = xml_tree.fromstring(esearch_assembly_resp.content).find('IdList')
            _Id_element = _IdList_element.find('Id') if _IdList_element is not None else None
            if _Id_element is None:
                raise NCBISearchNotFound(
                    search_term="Id (assembly)",
                    url=str(esearch_assembly_resp.url)
                )
            ncbi_assembly_id = _Id_element.text
            log.info(f"Found corresponding assembly Id {ncbi_assembly_id} on NCBI (assembly DB)")

        # Now summarize the assembly to get the Assembly Report FTP path
        esummary_assembly_resp = _get_with_retries(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi",
            params={"db": "assembly", "id": ncbi_assembly_id}
        )
        _DocumentSummarySet_element = xml_tree.fromstring(esummary_assembly_resp.content).find('DocumentSummarySet')
        if _DocumentSummarySet_element is None:
            raise NCBISearchNotFound(
                search_term="DocumentSummarySet",
                url=str(esummary_assembly_resp.url)
            )
        _DocumentSummary_element = _DocumentSummarySet_element.find("DocumentSummary")
        if _DocumentSummary_element is None:
            raise NCBISearchNotFound(
                search_term="DocumentSummary",
                url=str(esummary_assembly_resp.url)
            )
        _FtpPath_Assembly_rpt_element = _DocumentSummary_element.find("FtpPath_Assembly_rpt")
        if _FtpPath_Assembly_rpt_element is None or _FtpPath_Assembly_rpt_element.text is None:
            raise NCBISearchNotFound(
                search_term="FtpPath_Assembly_rpt",
                url=str(esummary_assembly_resp.url)
            )
        ncbi_assembly_report_url = _FtpPath_Assembly_rpt_element.text
        log.info(f"Downloading assembly information from: {ncbi_assembly_report_url}")

        # Load assembly report (tab-delimited; '#' comments)
        assembly_report = pd.read_csv(
            ncbi_assembly_report_url,
            comment='#',
            names=[
                "Sequence-Name", "Sequence-Role", "Assigned-Molecule",
                "Assigned-Molecule-Location/Type", "GenBank-Accn", "Relationship",
                "RefSeq-Accn", "Assembly-Unit", "Sequence-Length", "UCSC-style-name"
            ],
            sep='\t'
        )
        assembly_report = assembly_report.loc[
            assembly_report['Sequence-Role'] == 'assembled-molecule'
        ].copy()
        assembled_molecules = assembly_report['Sequence-Name'].to_list()
        str_assembled_molecules = "\n\t".join(assembled_molecules)
        log.info("Found following assembled molecules (chromosomes): \n\t" + str_assembled_molecules)

    except (MaxNCBIRetriesReached, NCBISearchNotFound) as e:
        print(e)
        print(
            "Returning gene annotation without subestting for assembled chromosomes"
            "and converting to UCSC style. Please make sure that the chromosome names"
            "in the returned object match with the chromosome names in the scplus_obj."
            "Chromosome sizes will not be returned"
        )
        return annot
    except Exception as e:
        print("Unhandeled exception occured")
        print(e)
        print(
            "Returning gene annotation without subestting for assembled chromosomes"
            "and converting to UCSC style. Please make sure that the chromosome names"
            "in the returned object match with the chromosome names in the scplus_obj."
            "Chromosome sizes will not be returned"
        )
        return annot
    else:
        # Subset to assembled chromosomes and build chromsizes
        annot = annot.query('Chromosome in @assembled_molecules').copy()
        chromsizes = pd.DataFrame(columns=["Chromosome", "Start", "End"])
        chromsizes["Start"] = np.repeat(0, len(assembly_report))
        chromsizes["End"] = assembly_report["Sequence-Length"].to_list()

        if use_ucsc_chromosome_style:
            ensembl_to_ucsc_chrom = assembly_report[['Sequence-Name', 'UCSC-style-name']] \
                .set_index('Sequence-Name')['UCSC-style-name'].to_dict()
            str_ensembl_to_ucsc_chrom = "\n\tOriginal\tUCSC"
            for ens in ensembl_to_ucsc_chrom.keys():
                str_ensembl_to_ucsc_chrom += "\n\t" + ens + "\t" + ensembl_to_ucsc_chrom[ens]
            log.info("Converting chromosomes names to UCSC style as follows: " + str_ensembl_to_ucsc_chrom)

            annot['Chromosome'] = [
                ensembl_to_ucsc_chrom.get(chrom, chrom) for chrom in annot['Chromosome']
            ]
            chromsizes["Chromosome"] = assembly_report["UCSC-style-name"].to_list()
        else:
            chromsizes["Chromosome"] = assembly_report["Sequence-Name"].to_list()

        return annot, chromsizes


In [14]:
from scenicplus.data_wrangling.gene_search_space import get_search_space
max_dist = 1000000

annot, chromsizes = download_gene_annotation_and_chromsizes(
        species = 'mmusculus',
        biomart_host='http://nov2020.archive.ensembl.org/',
    )

search_space = get_search_space(
        scplus_region = ad_atac.var_names.to_list(),
        scplus_genes = ad_rna.var_names.to_list(),
        gene_annotation = annot,
        chromsizes = chromsizes,
        use_gene_boundaries = False,
        upstream = (max_dist, max_dist),
        downstream = (max_dist, max_dist),
        extend_tss = (10, 10),
        remove_promoters = False
    )

2025-08-21 11:34:44,151 Download gene annotation INFO     Using genome: GRCm38.p6
2025-08-21 11:34:44,160 Download gene annotation INFO     No genome Id found for term 'GRCm38.p6'. Trying assembly DB…
2025-08-21 11:34:44,172 Download gene annotation INFO     Found corresponding assembly Id 1198761 on NCBI (assembly DB)
2025-08-21 11:34:44,183 Download gene annotation INFO     Downloading assembly information from: ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/001/635/GCF_000001635.26_GRCm38.p6/GCF_000001635.26_GRCm38.p6_assembly_report.txt
2025-08-21 11:35:15,002 Download gene annotation INFO     Found following assembled molecules (chromosomes): 
	1
	2
	3
	4
	5
	6
	7
	8
	9
	10
	11
	12
	13
	14
	15
	16
	17
	18
	19
	X
	Y
	MT
2025-08-21 11:35:15,014 Download gene annotation INFO     Converting chromosomes names to UCSC style as follows: 
	Original	UCSC
	1	chr1
	2	chr2
	3	chr3
	4	chr4
	5	chr5
	6	chr6
	7	chr7
	8	chr8
	9	chr9
	10	chr10
	11	chr11
	12	chr12
	13	chr13
	14	chr14
	15	chr15
	16	c

/data/groups/vib.ai/stein.aerts/gpartel/software/miniconda3/envs/deepSCENIC/lib/python3.11/site-packages/pyranges/methods/init.py:60: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return {k: v for k, v in df.groupby(grpby_key)}
/data/groups/vib.ai/stein.aerts/gpartel/software/miniconda3/envs/deepSCENIC/lib/python3.11/site-packages/pyranges/methods/init.py:60: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return {k: v for k, v in df.groupby(grpby_key)}


2025-08-21 11:35:15,788 Get search space INFO     Extending promoter annotation to 10 bp upstream and 10 downstream
2025-08-21 11:35:15,880 Get search space INFO     Extending search space to:
            						1000000 bp downstream of the end of the gene.
            						1000000 bp upstream of the start of the gene.


/data/groups/vib.ai/stein.aerts/gpartel/software/miniconda3/envs/deepSCENIC/lib/python3.11/site-packages/pyranges/methods/init.py:60: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return {k: v for k, v in df.groupby(grpby_key)}
/data/groups/vib.ai/stein.aerts/gpartel/software/miniconda3/envs/deepSCENIC/lib/python3.11/site-packages/pyranges/methods/init.py:60: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return {k: v for k, v in df.groupby(grpby_key)}
/data/groups/vib.ai/stein.aerts/gpartel/software/miniconda3/envs/deepSCENIC/lib/python3.11/site-packages/pyranges/methods/init.py:60: FutureWarning: The default

2025-08-21 11:35:16,229 Get search space INFO     Intersecting with regions.


/data/groups/vib.ai/stein.aerts/gpartel/software/miniconda3/envs/deepSCENIC/lib/python3.11/site-packages/pyranges/methods/init.py:60: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return {k: v for k, v in df.groupby(grpby_key)}
/data/groups/vib.ai/stein.aerts/gpartel/software/miniconda3/envs/deepSCENIC/lib/python3.11/site-packages/pyranges/methods/init.py:60: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return {k: v for k, v in df.groupby(grpby_key)}
/data/groups/vib.ai/stein.aerts/gpartel/software/miniconda3/envs/deepSCENIC/lib/python3.11/site-packages/pyranges/methods/init.py:187: FutureWarning: The defaul

2025-08-21 11:35:26,637 Get search space INFO     Calculating distances from region to gene


/data/groups/vib.ai/stein.aerts/gpartel/software/miniconda3/envs/deepSCENIC/lib/python3.11/site-packages/pyranges/methods/init.py:60: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return {k: v for k, v in df.groupby(grpby_key)}
/data/groups/vib.ai/stein.aerts/gpartel/software/miniconda3/envs/deepSCENIC/lib/python3.11/site-packages/scenicplus/utils.py:184: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values

2025-08-21 11:37:35,242 Get search space INFO     Imploding multiple entries per region and gene


/data/groups/vib.ai/stein.aerts/gpartel/software/miniconda3/envs/deepSCENIC/lib/python3.11/site-packages/pyranges/methods/init.py:60: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return {k: v for k, v in df.groupby(grpby_key)}


In [15]:
search_space['Abs_distance'] = np.array([abs(x[0]) + 1 for x in search_space['Distance'].to_list()]).astype(int)
search_space_subset = search_space.loc[[gene in ad_rna.var_names for gene in search_space['Gene']]]

Split chromosomes in train and test set 

In [16]:
# Subset regions and genes in test chromosomes
search_space_subset_test = search_space_subset[(search_space_subset.Name.str.startswith('chr19')) | (search_space_subset.Name.str.startswith('chr18')) | (search_space_subset.Name.str.startswith('chr11')) | (search_space_subset.Name.str.startswith('chr7'))]
regions_test = search_space_subset_test.loc[:, 'Name'].unique().tolist()
genes_test = search_space_subset_test.loc[:, 'Gene'].unique().tolist()

In [17]:
# Subset regions and genes in train chromosomes
search_space_subset_train = search_space_subset[~((search_space_subset.Name.str.startswith('chr19')) | (search_space_subset.Name.str.startswith('chr18')) | (search_space_subset.Name.str.startswith('chr11')) | (search_space_subset.Name.str.startswith('chr7')))]
regions_train = search_space_subset_train.loc[:, 'Name'].unique().tolist()
genes_train = search_space_subset_train.loc[:, 'Gene'].unique().tolist()

In [18]:
search_space_subset_test = search_space_subset_test[search_space_subset_test.Name.isin(regions_test)]
search_space_subset_train = search_space_subset_train[search_space_subset_train.Name.isin(regions_train)]

In [19]:
print(len(regions_train), len(regions_test))

438728 101899


In [20]:
# Function to apply as region distance prior

import math

def gaussian_function(x):
    mean=0
    std_dev = 100000
    exponent = -(1/2) * ((x - mean) / std_dev)**2
    return (1 / (std_dev * math.sqrt(2 * math.pi))) * math.exp(exponent)
def penalty_coeffs(x):
    return 1 - gaussian_function(x)/gaussian_function(0)

In [21]:
search_space_subset_train['distance_penalty'] = search_space_subset_train.Abs_distance.map(penalty_coeffs)
search_space_subset_test['distance_penalty'] = search_space_subset_test.Abs_distance.map(penalty_coeffs)

In [22]:
r2gpenalty_train = search_space_subset_train.pivot(index='Name', columns='Gene', values='distance_penalty')
r2gpenalty_test = search_space_subset_test.pivot(index='Name', columns='Gene', values='distance_penalty')
r2gpenalty_train = r2gpenalty_train.fillna(0)
r2gpenalty_test = r2gpenalty_test.fillna(0)

/data/tmp/330902/ipykernel_1766267/4086774179.py:1: PerformanceWarning: The following operation may generate 5625370416 cells in the resulting pandas object.
  r2gpenalty_train = search_space_subset_train.pivot(index='Name', columns='Gene', values='distance_penalty')


In [23]:
# Add missing TFs to the datasets
r2gpenalty_train.loc[:, TFs[~np.isin(TFs, r2gpenalty_train.columns)]] = 0
r2gpenalty_test.loc[:, TFs[~np.isin(TFs, r2gpenalty_test.columns)]] = 0

In [24]:
# Put TFs first
r2gpenalty_train = r2gpenalty_train.loc[:, TFs.to_list() + r2gpenalty_train.columns[~r2gpenalty_train.columns.isin(TFs)].to_list()]
r2gpenalty_test = r2gpenalty_test.loc[:, TFs.to_list() + r2gpenalty_test.columns[~r2gpenalty_test.columns.isin(TFs)].to_list()]

In [25]:
# Ensure same order of indices
regions_train = r2gpenalty_train.index.values.tolist()
genes_train = r2gpenalty_train.columns.values.tolist()

regions_test = r2gpenalty_test.index.values.tolist()
genes_test = r2gpenalty_test.columns.values.tolist()

ad_atac_train = ad_atac[:, regions_train].copy()
ad_rna_train = ad_rna[:, genes_train].copy()
ad_atac_test = ad_atac[:, regions_test].copy()
ad_rna_test = ad_rna[:, genes_test].copy()

In [26]:
from scipy.sparse import coo_array, save_npz
r2gpenalty_coo_train = coo_array(r2gpenalty_train)
r2gpenalty_coo_test = coo_array(r2gpenalty_test)

In [37]:
ad_rna = sc.concat((ad_rna_train, ad_rna_test[:, ~ad_rna_test.var_names.isin(ad_rna_train.var_names)]), axis=1)
ad_rna.obs = ad_rna_train.obs
ad_rna

AnnData object with n_obs × n_vars = 39470 × 16437
    obs: 'nCount_RNA', 'nFeature_RNA', 'nCount_ATAC', 'nFeature_ATAC', 'nCount_SCT', 'nFeature_SCT', 'class_Bakken_2022', 'subclass_Bakken_2022', 'cluster_Bakken_2022', 'species', 'sample'
    var: 'gene', 'n_cells'

In [38]:
ad_atac = sc.concat((ad_atac_train, ad_atac_test), axis=1)
ad_atac.obs = ad_atac_train.obs
ad_atac

AnnData object with n_obs × n_vars = 39470 × 540627
    obs: 'cisTopic_nr_frag', 'cisTopic_log_nr_frag', 'cisTopic_nr_acc', 'cisTopic_log_nr_acc', 'sample_id', 'Unique_nr_frag', 'barcode', 'pycisTopic_leiden_10_0.6', 'pycisTopic_leiden_10_1.2', 'cluster_Bakken_2022', 'class_Bakken_2022', 'subclass_Bakken_2022'
    var: 'Chromosome', 'Start', 'End', 'Width', 'cisTopic_nr_frag', 'cisTopic_log_nr_frag', 'cisTopic_nr_acc', 'cisTopic_log_nr_acc'

Split cells in train and test

In [39]:
from sklearn.model_selection import train_test_split
train_idxs, test_idxs = train_test_split(ad_rna_train.obs.index, test_size=0.2, random_state=42)

In [40]:
ad_atac_train = ad_atac_train[train_idxs]
ad_rna_train = ad_rna_train[train_idxs]
ad_atac_test = ad_atac_test[test_idxs]
ad_rna_test = ad_rna_test[test_idxs]

In [41]:
ad_rna_E2 = ad_rna[ad_rna_train.obs.index, ad_rna_test.var_names]
ad_atac_E2 = ad_atac[ad_atac_train.obs.index, ad_atac_test.var_names]

Store DARs (computed with pyCisTopic)

In [42]:
from scenicplus.utils import region_names_to_coordinates, coord_to_region_names
import pyranges as pr

DARS_DATA_PATH = '/data/groups/vib.ai/stein.aerts/gpartel/data/mouse_biccn/DARs/'

ct_name_d = {
    'L6IT':'L6 IT',
    'Vip':'Vip',
    'L5-6NP':'L5/6 NP',
    'Pvalb':'Pvalb',
    'L5IT':'L5 IT',
    'Sst':'Sst',
    'Endo':'Endo',
    'L6b':'L6b',
    'Sncg':'Sncg',
    'OPC':'OPC',
    'SstChodl':'Sst Chodl',
    'VLMC':'VLMC',
    'Lamp5':'Lamp5',
    'L6ITCar3':'L6 IT Car3',
    'Astro':'Astro',
    'Oligo':'Oligo',
    'L5ET':'L5 ET',
    'L2-3IT':'L2/3 IT',
    'L6CT':'L6 CT',
    'Micro-PVM':'Micro-PVM'
}   

DARS_d = {}
for f in os.listdir(DARS_DATA_PATH):
    ct = f.split('.')[0]

    df = pr.PyRanges(pd.read_csv(DARS_DATA_PATH + f, header=None, sep='\t', names=['Chromosome', 'Start', 'End']))
    DARS_d[ct_name_d[ct]] = coord_to_region_names(df)

/data/groups/vib.ai/stein.aerts/gpartel/software/miniconda3/envs/deepSCENIC/lib/python3.11/site-packages/pyranges/methods/init.py:60: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return {k: v for k, v in df.groupby(grpby_key)}
/data/groups/vib.ai/stein.aerts/gpartel/software/miniconda3/envs/deepSCENIC/lib/python3.11/site-packages/pyranges/methods/init.py:60: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return {k: v for k, v in df.groupby(grpby_key)}
/data/groups/vib.ai/stein.aerts/gpartel/software/miniconda3/envs/deepSCENIC/lib/python3.11/site-packages/pyranges/methods/init.py:60: FutureWarning: The default

In [43]:
ad_atac.var.loc[:, 'dar'] = 0

DARs = []
for ct in DARS_d.keys():
    regs = list(np.intersect1d(DARS_d[ct], ad_atac.var_names))
    DARs.extend(regs)

DARs = list(set(DARs))
ad_atac.var.loc[DARs, 'dar'] = 1

In [ ]:
# Saving files
ad_rna.write(DATA_PATH+'raw_exprMat.h5ad')
ad_atac.write(DATA_PATH+'fragment_matrix.h5ad')
ad_rna_train.write(DATA_PATH+'raw_exprMat_train.h5ad')
ad_atac_train.write(DATA_PATH+'fragment_matrix_train.h5ad')
ad_rna_test.write(DATA_PATH+'raw_exprMat_test.h5ad')
ad_atac_test.write(DATA_PATH+'fragment_matrix_test.h5ad')
ad_rna[ad_rna_test.obs.index, ad_rna_train.var_names].write(DATA_PATH+'raw_exprMat_train_eval.h5ad')
ad_atac_E2.write(DATA_PATH+'fragment_matrix_E2.h5ad')
ad_rna_E2.write(DATA_PATH+'raw_exprMat_E2.h5ad')
ad_atac[ad_atac_test.obs.index, ad_atac_train.var_names].write(DATA_PATH+'fragment_matrix_ctest_rtrain.h5ad')